In [8]:
from anthropic import Anthropic, APIError
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import Any, Dict, List

load_dotenv()

client = Anthropic()
model = "claude-sonnet-5"

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return next(block.text for block in message.content if block.type == "text")

# claude-sonnet-5 doesn't support assistant message prefill, so the old
# prefill("```json") + stop_sequence("```") trick to force clean JSON is
# flaky here (sometimes StopIteration). Use structured outputs instead,
# which guarantees valid JSON without prefill or stop sequences.
class EventBridgeRule(BaseModel):
    source: List[str]
    detail_type: List[str]
    event_pattern: Dict[str, Any]

messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")

response = client.messages.parse(
    model=model,
    max_tokens=1000,
    messages=messages,
    output_format=EventBridgeRule,
)

print(response.parsed_output.model_dump_json(indent=2))

{
  "source": [
    "aws.ec2"
  ],
  "detail_type": [
    "EC2 Instance State-change Notification"
  ],
  "event_pattern": {}
}
